# Autonomous Surface Crack Detection and Segmentation using YOLO11-Seg
### *A Deep Learning Framework for Structural Health Monitoring and Surface Defect Quantification*

---

## 1. Abstract & Introduction
Structural health monitoring (SHM) is critical for ensuring the safety and longevity of infrastructure such as bridges, roads, and concrete structures. Traditional manual inspection of surface cracks is time-consuming, subjective, and prone to error. Recent advances in deep learning, particularly object detection and instance segmentation models, have enabled rapid and automated inspection.

While standard object detection models output bounding boxes, they are insufficient for detailed crack characterization. In contrast, **instance segmentation** provides pixel-level masks, allowing engineering teams to:
1. Measure the exact **area** and **perimeter** of the crack.
2. Estimate the **crack length** and **average width (aperture)** by skeletonizing the segmented masks.
3. Track **crack propagation and orientation** over time.

This notebook demonstrates the complete end-to-end pipeline to train, validate, and deploy an **Ultralytics YOLO11 Instance Segmentation model** (`yolo11n-seg`) on a concrete crack segmentation dataset.

### Technical Workflow
1. **Environment Setup & Verification**: Confirm hardware acceleration (CUDA GPU) and install necessary packages.
2. **Exploratory Data Analysis (EDA)**: Programmatically check dataset splits and visualize raw images with annotated ground-truth segmentations.
3. **Model Initialization**: Load a pretrained YOLO11-seg model.
4. **Training Optimization**: Configure hyperparameters (batch size, epochs, patience, optimizer) and initiate training on the local dataset.
5. **Quantitative Validation**: Evaluate model performance on the validation set using Precision, Recall, and mean Average Precision (mAP) for both bounding boxes and segmentation masks.
6. **Qualitative Inference**: Run inference on unseen test images and plot side-by-side comparative visualizations.
7. **Model Serialization**: Export the trained PyTorch weights to ONNX format for edge deployment.
8. **Real-time Deployment**: Run inference using camera/webcam feeds with overlay mask rendering.

## 2. System Environment and Dependencies
First, we verify the environment, ensure CUDA is configured for GPU acceleration, and check the installed versions of PyTorch and the `ultralytics` package.

In [ ]:
import sys
import subprocess
from pathlib import Path
import torch

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: CUDA GPU not detected. Running training on CPU will be significantly slower.")

# Constants & Paths
DATASET_DIR = Path(r"D:\All Code\New folder\crack_detect")
DATA_YAML = DATASET_DIR / "crack-seg.yaml"
RUN_NAME = "crack_seg_train"
PROJECT_DIR = DATASET_DIR / "runs" / "segment" / "runs" / "segment"

try:
    import ultralytics
    print(f"Ultralytics YOLO Version: {ultralytics.__version__}")
except ImportError:
    print("Installing Ultralytics...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-U", "ultralytics"], check=True)
    import ultralytics
    print(f"Ultralytics YOLO Version: {ultralytics.__version__}")

print(f"Workspace Dataset Directory: {DATASET_DIR}")
print(f"Dataset YAML Path: {DATA_YAML}")

## 3. Dataset Architecture & Exploratory Data Analysis (EDA)
In instance segmentation, annotations are stored as normalized polygon coordinate pairs rather than bounding boxes. We verify the presence of the training, validation, and testing sets, count the samples, and then display raw images overlaid with their ground-truth polygon masks.

### Mathematical Definition of Bounding Box vs. Segmentation Mask:
- A bounding box $B$ is represented by four coordinates: $B = [x_{center}, y_{center}, width, height]$.
- A segmentation mask $M$ is a set of polygon coordinates: $M = \{(x_1, y_1), (x_2, y_2), \dots, (x_N, y_N)\}$ defining the boundary of the object.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
splits = ["train", "val", "test"]

# Verify Paths
assert DATASET_DIR.exists(), f"Dataset directory not found: {DATASET_DIR}"
assert DATA_YAML.exists(), f"Dataset configuration file not found: {DATA_YAML}"

# Check dataset counts
print("=" * 60)
print(f"{'Data Split':<12} | {'Images':<8} | {'Labels':<8} | {'Missing Labels':<14}")
print("-" * 60)

for split in splits:
    image_dir = DATASET_DIR / "images" / split
    label_dir = DATASET_DIR / "labels" / split
    assert image_dir.exists(), f"Missing image directory for {split}: {image_dir}"
    assert label_dir.exists(), f"Missing label directory for {split}: {label_dir}"

    images = sorted(p for p in image_dir.iterdir() if p.suffix.lower() in image_exts)
    labels = sorted(label_dir.glob("*.txt"))
    missing = [p.name for p in images if not (label_dir / f"{p.stem}.txt").exists()]
    
    print(f"{split:<12} | {len(images):<8} | {len(labels):<8} | {len(missing):<14}")

# Visualizing Sample Images with Masks
def draw_yolo_masks(image_path, label_path):
    img = cv2.imread(str(image_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, _ = img.shape
    
    mask_overlay = img.copy()
    
    if label_path.exists():
        with open(label_path, "r") as f:
            lines = f.readlines()
        for line in lines:
            parts = list(map(float, line.strip().split()))
            if not parts:
                continue
            class_id = int(parts[0])
            # Coords are normalized x y pairs
            coords = np.array(parts[1:]).reshape(-1, 2)
            coords[:, 0] *= w
            coords[:, 1] *= h
            coords = coords.astype(np.int32)
            
            # Draw polygon
            cv2.fillPoly(mask_overlay, [coords], color=(255, 0, 0)) # Red mask for cracks
            cv2.polylines(img, [coords], isClosed=True, color=(255, 255, 0), thickness=2) # Yellow border

    # Blend original and mask overlay
    blended = cv2.addWeighted(mask_overlay, 0.4, img, 0.6, 0)
    return blended

# Display 4 samples from the training set
train_image_dir = DATASET_DIR / "images" / "train"
train_label_dir = DATASET_DIR / "labels" / "train"
sample_images = sorted(p for p in train_image_dir.iterdir() if p.suffix.lower() in image_exts)[:4]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, img_path in enumerate(sample_images):
    lbl_path = train_label_dir / f"{img_path.stem}.txt"
    visualized = draw_yolo_masks(img_path, lbl_path)
    axes[i].imshow(visualized)
    axes[i].set_title(f"Sample {i+1}: {img_path.name}")
    axes[i].axis("off")

plt.tight_layout()
plt.suptitle("Exploratory Data Analysis: Ground Truth Annotation Visualizations", y=1.02, fontsize=16, fontweight='bold')
plt.show()

## 4. Model Configuration & Transfer Learning
We load the pretrained `yolo11n-seg.pt` model weights. Since instance segmentation requires localizing object boundaries, using a model pre-trained on the COCO dataset provides robust lower-level feature extractors (like edge and texture detectors) that accelerate convergence.

### Hyperparameters Overview:
- `epochs`: 50 (Determines the number of passes over the dataset).
- `batch`: 8 (Selected to balance memory limits and optimization stability).
- `imgsz`: 640 (Input resolution. Images will be resized while maintaining aspect ratio).
- `device`: GPU acceleration `0` (if CUDA is available) or `'cpu'` as fallback.
- `patience`: 20 (Early stopping patience to prevent overfitting).

In [ ]:
from ultralytics import YOLO

# Model Weight Configuration
MODEL_TYPE = "yolo11n-seg.pt"
EPOCHS = 50
IMGSZ = 640
BATCH_SIZE = 8

# Initialize YOLO model with pretrained segmentation weights
model = YOLO(MODEL_TYPE)

print(f"Successfully loaded model backbone: {MODEL_TYPE}")

# Note: The training process will output status logs for each epoch.
# We set exist_ok=True so that re-running does not create nested directories.
results = model.train(
    data=str(DATA_YAML),
    task="segment",
    imgsz=IMGSZ,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    project=str(PROJECT_DIR.parent), # Top level directory runs/segment
    name=RUN_NAME,
    exist_ok=True,
    patience=20,
    device=0 if torch.cuda.is_available() else 'cpu',
)

## 5. Quantitative Model Validation
Validation measures how well our model generalizes to unseen data. In object detection and segmentation, we use mean Average Precision (mAP) computed at different Intersection over Union (IoU) thresholds.

### Key Metrics Defined:
1. **Precision (P)**: The proportion of correct positive predictions relative to all positive predictions:
   $$Precision = \frac{TP}{TP + FP}$$
2. **Recall (R)**: The proportion of correct positive predictions relative to all actual positive instances:
   $$Recall = \frac{TP}{TP + FN}$$
3. **mAP@50**: Mean Average Precision computed at an Intersection over Union (IoU) threshold of 0.5.
4. **mAP@50-95**: Mean Average Precision computed across a sequence of IoU thresholds ranging from 0.5 to 0.95 with a step of 0.05. This represents a more rigorous evaluation of segmentation localization quality.

In [ ]:
import pandas as pd

# Load validation weights
BEST_WEIGHTS = PROJECT_DIR / RUN_NAME / "weights" / "best.pt"
assert BEST_WEIGHTS.exists(), f"Trained weights not found at: {BEST_WEIGHTS}. Please complete model training."

# Load the model with trained weights
best_model = YOLO(str(BEST_WEIGHTS))

# Execute validation
metrics = best_model.val(data=str(DATA_YAML), split="val", imgsz=IMGSZ)

print("\n" + "=" * 50)
print("             VALIDATION METRICS SUMMARY")
print("=" * 50)
print(f"Box Precision (B)      : {metrics.results_dict['metrics/precision(B)']:.4f}")
print(f"Box Recall (B)         : {metrics.results_dict['metrics/recall(B)']:.4f}")
print(f"Box mAP@50 (B)         : {metrics.results_dict['metrics/mAP50(B)']:.4f}")
print(f"Box mAP@50-95 (B)      : {metrics.results_dict['metrics/mAP50-95(B)']:.4f}")
print("-" * 50)
print(f"Mask Precision (M)     : {metrics.results_dict['metrics/precision(M)']:.4f}")
print(f"Mask Recall (M)        : {metrics.results_dict['metrics/recall(M)']:.4f}")
print(f"Mask mAP@50 (M)        : {metrics.results_dict['metrics/mAP50(M)']:.4f}")
print(f"Mask mAP@50-95 (M)     : {metrics.results_dict['metrics/mAP50-95(M)']:.4f}")
print("=" * 50)

# Load and Plot results.csv if it exists
results_csv_path = PROJECT_DIR / RUN_NAME / "results.csv"
if results_csv_path.exists():
    df = pd.read_csv(results_csv_path)
    # Clean column headers (strip whitespaces)
    df.columns = df.columns.str.strip()
    
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot Losses
    ax[0].plot(df['epoch'], df['train/box_loss'], label='Train Box Loss', color='blue')
    ax[0].plot(df['epoch'], df['train/seg_loss'], label='Train Seg Loss', color='cyan')
    ax[0].plot(df['epoch'], df['val/box_loss'], label='Val Box Loss', linestyle='--', color='darkblue')
    ax[0].plot(df['epoch'], df['val/seg_loss'], label='Val Seg Loss', linestyle='--', color='darkcyan')
    ax[0].set_title('Training and Validation Loss')
    ax[0].set_xlabel('Epoch')
    ax[0].set_ylabel('Loss')
    ax[0].legend()
    ax[0].grid(True)
    
    # Plot mAP Metrics
    ax[1].plot(df['epoch'], df['metrics/mAP50(B)'], label='Box mAP@50', color='red')
    ax[1].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='Box mAP@50-95', color='salmon')
    ax[1].plot(df['epoch'], df['metrics/mAP50(M)'], label='Mask mAP@50', color='green')
    ax[1].plot(df['epoch'], df['metrics/mAP50-95(M)'], label='Mask mAP@50-95', color='lightgreen')
    ax[1].set_title('Mean Average Precision (mAP) History')
    ax[1].set_xlabel('Epoch')
    ax[1].set_ylabel('mAP Value')
    ax[1].legend()
    ax[1].grid(True)
    
    plt.suptitle("Quantitative Training Dashboard", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 6. Qualitative Evaluation (Test Set Predictions)
To verify visual quality, we perform batch inference on the testing split (unseen images) and plot predictions side-by-side with original images, drawing both bounding boxes and filled masks.

In [ ]:
TEST_DIR = DATASET_DIR / "images" / "test"
test_images = sorted(TEST_DIR.glob("*.jpg"))[:4]
assert test_images, f"No test images found in {TEST_DIR}"

# Run predictions
pred_results = best_model.predict(
    source=[str(p) for p in test_images],
    conf=0.25,
    imgsz=IMGSZ,
    save=False, # We plot in the notebook
    verbose=False,
)

fig, axes = plt.subplots(len(test_images), 2, figsize=(12, 4 * len(test_images)))

for i, res in enumerate(pred_results):
    orig_img = res.orig_img.copy()
    orig_img = cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB)
    
    # Plot Original
    axes[i, 0].imshow(orig_img)
    axes[i, 0].set_title(f"Original: {Path(res.path).name}")
    axes[i, 0].axis("off")
    
    # Plot Prediction
    # res.plot() returns BGR image with masks and labels
    pred_img = res.plot(labels=True, boxes=True, masks=True)
    pred_img = cv2.cvtColor(pred_img, cv2.COLOR_BGR2RGB)
    
    axes[i, 1].imshow(pred_img)
    axes[i, 1].set_title(f"YOLO11 Prediction (Mask & Bbox)")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.suptitle("Comparative Qualitative Inference: Original vs. Predicted Masks", y=1.01, fontsize=16, fontweight='bold')
plt.show()

## 7. Model Exportation (ONNX Format)
For real-world deployment (e.g. inside browser runtimes, mobile applications, or embedded edge computers), we export the PyTorch model weights (`.pt`) to the **ONNX (Open Neural Network Exchange)** format.

In [ ]:
# Export model to ONNX format with graph optimization
try:
    onnx_path = best_model.export(format="onnx", imgsz=IMGSZ, simplify=True)
    print(f"Model successfully exported to ONNX format at:\n{onnx_path}")
except Exception as e:
    print(f"ONNX Export failed: {e}")

## 8. Real-time Edge Deployment (Webcam Video Stream)
This block enables running real-time crack segmentation via the local webcam interface. The model processes video frames sequentially and renders bounding boxes, class labels, and transparency masks.
Press `q` within the popup video window to gracefully exit the stream.

In [ ]:
# Change WEBCAM_SOURCE if you have multiple cameras (e.g. 1, 2)
WEBCAM_SOURCE = 0

# Note: This block will run a local window. If executed in a headless server environment,
# ensure a virtual frame buffer is configured.
try:
    print(f"Starting real-time inference on source {WEBCAM_SOURCE}. Press 'q' on the camera window to quit.")
    best_model.predict(
        source=WEBCAM_SOURCE,
        conf=0.25,
        imgsz=IMGSZ,
        show=True,
        save=False,
        verbose=False,
    )
except Exception as e:
    print(f"Webcam stream interrupted or not supported in this environment: {e}")